In [1]:
import numpy as np
import pytz

from Data_query.trino_config import *
from visualisation import *

In [5]:
stop_trino()

Trino service stopping triggered.


In [3]:
sleep(120)

In [5]:
sleep(180)

In [6]:
big_workers = 1
workers = 0
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count=workers, big_worker_desired_count=big_workers)
sleep(60)

Trino service is not running. Starting the service...
Trino service triggered.
Service trino-service is now stable.


In [7]:
iceberg_exec("DROP TABLE IF EXISTS all_uncurtailedPV")
iceberg_exec("""CREATE TABLE all_uncurtailedPV (
                site_id BIGINT,
                t_stamp TIMESTAMP,
                year INT,
                month INT,
                uncurtailed_P DOUBLE,
                P_kw DOUBLE,
                GHI DOUBLE,
                n_train BIGINT
            )
             WITH (
                format = 'PARQUET',
                partitioning = ARRAY['year', 'month'],
                sorted_by = ARRAY['site_id', 't_stamp']
             )
    """)

Executed
Executed


In [10]:
df0 = iceberg_sql("""select  a.site_id, max(uncurtailed_P - S_99) as max_diff, max(S_99) as S_99, max(ac_capacity_kw) as ac_capacity_kw
            from  all_uncurtailedPV a left join (select distinct site_id, S_99, ac_capacity_kw from meta_up23c) as m on a.site_id = m.site_id
            where uncurtailed_P - S_99 > .5
            group by a.site_id
            order by max_diff desc""")

In [21]:
df0

,site_id,max_diff,S_99,ac_capacity_kw
0,146075438,591.720295,17.332262,25.0
1,1245551345,130.268327,8.277151,20.0
2,929712738,43.557779,19.437275,20.0
3,1578875240,41.798883,99.831540,109.0
4,1150568841,39.218963,2.955021,4.0
...,...,...,...,...
6528,246361091,0.500374,5.001917,5.0
6529,1092005111,0.500284,4.951891,5.0
6530,517044628,0.500208,4.980803,5.0
6531,2090445270,0.500097,2.985170,3.0


In [18]:
df1 = iceberg_sql("""select  a.site_id, max(uncurtailed_P - S_99) as max_diff, max(S_99) as S_99, max(ac_capacity_kw) as ac_capacity_kw
            from  all_uncurtailedPV a left join (select distinct site_id, S_99, ac_capacity_kw from meta_up23c) as m on a.site_id = m.site_id
            where uncurtailed_P - S_99 > .5
            group by a.site_id
            order by max_diff desc""")

In [20]:
df1

,site_id,max_diff,S_99,ac_capacity_kw
0,146075438,591.720295,17.332262,25.0
1,1245551345,130.268327,8.277151,20.0
2,1578875240,44.725161,99.831540,109.0
3,929712738,43.165293,19.437275,20.0
4,1150568841,39.218963,2.955021,4.0
...,...,...,...,...
6730,975509130,0.500327,7.491396,8.2
6731,726418804,0.500275,5.885630,6.0
6732,651381746,0.500246,8.454816,8.0
6733,862820921,0.500011,2.974822,3.0


In [3]:
iceberg_sql("""select  *
            from  all_uncurtailedPV a left join (select distinct site_id, S_99, ac_capacity_kw from meta_up23c) as m on a.site_id = m.site_id
            where a.site_id = 146075438 
            and uncurtailed_p > 200
            limit 10""")

,site_id,t_stamp,year,month,uncurtailed_p,ghi_norm,n_train,site_id,S_99,ac_capacity_kw


In [3]:
iceberg_sql("""select  * 
            from pv_ghi_model
            where site_id=1792599725
            order by abs(a) desc""")

,site_id,tod_bin,a,b,n
0,1792599725,05:45:00,-3.816519,0.025876,7
1,1792599725,05:50:00,0.943541,0.000303,11
2,1792599725,16:55:00,0.653360,0.000802,30
3,1792599725,17:00:00,0.598229,0.000929,29
4,1792599725,16:35:00,0.527609,0.000927,54
...,...,...,...,...,...
130,1792599725,11:00:00,0.029856,0.000881,166
131,1792599725,10:55:00,0.021432,0.000889,170
132,1792599725,06:25:00,-0.014654,0.002846,85
133,1792599725,06:35:00,-0.003371,0.002649,72


In [ ]:
iceberg_sql("""select  * 
            from split_days
            where site_id = 146075438
            and day_type = 'train'
            limit 100""")

,site_id,actual_day,day_type
0,146075438,2024-09-03,train
1,146075438,2024-05-11,train
2,146075438,2024-04-30,train
3,146075438,2024-06-16,train
4,146075438,2024-06-14,train
...,...,...,...
95,146075438,2024-05-20,train
96,146075438,2024-10-12,train
97,146075438,2024-06-24,train
98,146075438,2024-05-17,train


In [8]:
sleep(20)

In [9]:
num_parts = 3
time_bin_interval = "5"  # in minutes
model = "pv_ghi_norm_model"
acceptible_sites = ", ".join(
    map(str, pd.read_csv("mape<50_sites.csv")["site_id"].tolist())
)


def run_func(args):
    year, month, part = args
    # time_filter = f"year = {year} and month = {month}"
    time_filter = f"year = {year}"
    part_filter = f"site_id % {num_parts} = {part}"
    df = iceberg_exec(f"""
                    insert into all_uncurtailedPV
                    with eligible_data AS (
                        SELECT
                            site_id,
                            actual_day,
                            t_stamp,
                            CAST(
                                date_trunc('minute', t_stamp + interval '10' hour)
                                - interval '1' minute * (minute(t_stamp + interval '10' hour) % {time_bin_interval})
                                AS TIME) AS tod_bin,
                            GHI/GHI_cs AS x,
                            GHI_cs,
                            P_kw_norm/ NULLIF(P_kw_norm_cs, 0.0) AS y,
                                P_kw_norm,
                                P_kw_norm_cs,
                                S_norm,
                                S_99,
                                V
                        FROM structured_data
                        WHERE P_kw_norm_cs > 0.2 AND GHI > 50 and P_kw_norm > 0.05 and P_kw_norm <= P_kw_norm_cs
                            and {time_filter} and {part_filter} and site_id in ({acceptible_sites})
                    ),
                    validation_on_eligible_data AS (
                        select 
                            t.site_id, 
                            t.t_stamp, 
                            x as GHI,
                            P_kw_norm, 
                            case when P_kw_norm_cs * (a + b * x) >= P_kw_norm then P_kw_norm_cs * (a + b * x) else P_kw_norm end AS P_kw_norm_est,
                            V,
                            S_norm,
                            S_99, 
                            m.n as n_train
                        from eligible_data t 
                            join {model} m on t.site_id = m.site_id and t.tod_bin = m.tod_bin
                    )
                    SELECT site_id, t_stamp, year(t_stamp) as year, month(t_stamp) as month, P_kw_norm_est*S_99 as uncurtailed_P, P_kw_norm*S_99, GHI, n_train 
                    FROM validation_on_eligible_data
                        where P_kw_norm_est is not null
                        """)

    #

    # sleep(10)
    print(f"Completed {time_filter},  part {part}")
    return df


tasks = [
    (year, month, part)
    for year in (2024,)
    for month in range(1, 2)
    for part in range(0, num_parts)
]
#   for split_cons in ['system.bucket(postcode, 16) > -1'] ]

try:
    df = trino_parallel_batch(
        run_func, tasks, num_workers=num_workers, batch_size=num_workers
    )
except Exception as e:
    print(f"Error during data retrieval: {e}")
finally:
    # stop_trino()
    pass
# df['t_stamp'] = pd.to_datetime(df['t_stamp']).dt.tz_localize('utc').dt.tz_convert(pytz.FixedOffset(10*60))
df

Executed
Completed year = 2024,  part 0
Executed
Completed year = 2024,  part 1
Executed
Completed year = 2024,  part 2


In [12]:
iceberg_sql("select * from all_uncurtailedPV where uncurtailed_P < P_kw limit 10")

,site_id,t_stamp,year,month,uncurtailed_p,p_kw,ghi,n_train
